In [1]:
!pip install -q ipywidgets
!pip install -q beautifulsoup4
!pip install -q networkx

In [33]:
import json
import os
import pickle
import re

# data preprocessing
from collections import defaultdict
from io import StringIO
from itertools import combinations
from urllib.parse import quote

import matplotlib.pyplot as plt

# networks
import networkx as nx

# from networkx.readwrite import gpickle
import pandas as pd

# webscraping
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# Collect the Team Composition data through a rest API

In [34]:
with open("data/json_data.txt", "r") as file:
    content = file.read()
    data = json.loads(content)

In [35]:
def create_mapping():
    """Creates a mapping from avatar image URLs to English character names."""
    image_english_mapping = {}

    for char_info in data["has_list"]:
        name = char_info["name"]
        image_link = char_info["avatar"]

        # Match Chinese name with English equivalent
        cn_to_ens = data["select_list"]
        for cn_to_en in cn_to_ens:
            cn = cn_to_en["title"][:-2]
            if cn == name:
                en = cn_to_en["value"]
                image_english_mapping[image_link] = en
                break  # Stop once matched

    # Add default mapping for Traveler
    image_english_mapping.update(
        {
            (
                "https://upload-bbs.mihoyo.com/game_record/genshin/character_icon/"
                "UI_AvatarIcon_PlayerGirl.png"
            ): "Traveler"
        }
    )

    # Add nefer too
    image_english_mapping.update(
        {
            (
                "https://inews.gtimg.com/om_bt/OZBabi2uXPQ10bwzQ6OP9abQ0YQvcyaSmMYn4UWERmFooAA/0"
            ): "Nefer"
        }
    )

    return image_english_mapping


image_english_mapping = create_mapping()

In [36]:
# helper function
def get_character_names():
    """Get the character names"""
    url = "https://api.yshelper.com/ys/getAbyssRank.php"
    data = requests.get(url).json()

    char_names = []
    for char_json in data["select_list"][1:]:
        char_names.append(char_json["value"])

    return char_names


char_names = get_character_names()

In [37]:
# helper function
def get_team_usage_per_char(char_name, version):
    """
    Given the character name and version, return the top 100 or fewer
    teams that include the character (from result[3]).
    """
    base_url = "https://api.yshelper.com/ys/getAbyssRank.php"
    url = f"{base_url}?star=all&role={quote(char_name)}" f"&lang=en&version={version}"

    team_data = requests.get(url).json()["result"][3]

    return team_data if team_data else None

In [38]:
# important function
def get_team_per_version(version, char_names):
    """Given the version, get all the possible teams in a list
    with each elemenet as a json str"""
    possible_teams = set()

    # loop over all the characters
    for char in tqdm(char_names, desc="Fetching team usage"):
        # get the teams
        char_team_usage = get_team_usage_per_char(char, version)

        # if there are no teams, then skip this character
        if not char_team_usage:
            continue

        # add to a set all the possible teams
        for team in char_team_usage:
            possible_teams.add(json.dumps(team, sort_keys=True))
    return list(possible_teams)

In [39]:
# important function
def get_occur_version(teams, image_english_mapping):
    """
    Compute normalized pairwise character co-occurrence rates for a specific game version.

    Parameters
    ----------
    teams : iterable
        A collection of team JSON strings. Each string represents a team and contains:
        - "role": a list of dictionaries where each entry includes an "avatar" field
        - "attend_rate": a float indicating how often that team appears in the dataset

    image_english_mapping : dict
        A dictionary mapping avatar image filenames to English character names.
        Only characters that successfully map are included in the co-occurrence pairs.

    Returns
    -------
    dict
        A dictionary where keys are tuples (char1, char2) in alphabetical order,
        and values are the accumulated normalized co-occurrence rates based on
        each team's attend_rate. For example:
            {
                ("Ayaka", "Shenhe"): 0.184,
                ("Bennett", "Xiangling"): 0.452,
                ...
            }

    Notes
    -----
    - A team's characters are extracted from `team['role']`.
    - Pair combinations are formed using all unique sorted pairs of mapped characters.
    - Each pair's value increases by the team's attend_rate.
    - tqdm is used to display progress for large team lists.
    """
    co_occur = defaultdict(float)
    for team in tqdm(teams, desc="Getting the normalized co-occurence rate"):
        # get the names
        team = json.loads(team)
        char_names = []
        for character_dict in team["role"]:
            character = character_dict["avatar"]
            char_name = image_english_mapping[character]
            if char_name:  # Only add if mapping is successful
                char_names.append(char_name)

        # get each character combination
        for char1, char2 in combinations(sorted(char_names), 2):
            co_occur[(char1, char2)] += team["attend_rate"]

    return co_occur

In [40]:
# important function
def get_graph(co_occur):
    """
    Create a NetworkX graph from pairwise character co-occurrence data.

    Parameters
    ----------
    co_occur : dict
        A dictionary where keys are tuples (char1, char2) and values are
        normalized co-occurrence weights. For example:
            {
                ("Ayaka", "Shenhe"): 1.84,
                ("Bennett", "Xiangling"): 3.12,
                ...
            }

    Returns
    -------
    networkx.Graph
        An undirected graph where:
        - Each node is a character.
        - Each edge represents co-occurrence between two characters.
        - Edges include a 'weight' attribute equal to the co-occurrence value.

    Notes
    -----
    - Pairs are sorted in descending order before filtering.
    - The graph is undirected since co-occurrence is symmetric.
    """
    sorted_list = sorted(co_occur.items(), key=lambda x: x[1], reverse=True)
    filtered_list = [item for item in sorted_list if item[1]]

    co_occur_graph = nx.Graph()
    for char_comb, weight in filtered_list:
        char1, char2 = char_comb[0], char_comb[1]
        co_occur_graph.add_edge(char1, char2, weight=co_occur[(char1, char2)])

    return co_occur_graph

In [41]:
# main function
def create_graph(
    version, char_names=char_names, image_english_mapping=image_english_mapping
):
    """
    Generate a co-occurrence graph for a specific game version.

    This function orchestrates all major steps:
    1. Fetch team compositions for the given version.
    2. Compute normalized co-occurrence weights for all character pairs.
    3. Build a NetworkX graph representing these co-occurrences.

    Parameters
    ----------
    version : int or str
        The game version to analyze. This is passed to the webscraper that
        retrieves teams used in that version.
    char_names : list of str, optional
        List of all character names to iterate over when scraping team data.
        Defaults to the globally defined `char_names`.
    image_english_mapping : dict, optional
        Mapping from internal avatar identifiers to English character names.
        Used to translate scraped data into standardized character names.

    Returns
    -------
    networkx.Graph
        A co-occurrence graph where:
        - Nodes are characters.
        - Edges represent pairwise co-occurrence.
        - Edge weights correspond to normalized co-occurrence values.
        Only pairs meeting the weight criteria from `get_graph` are included.

    Notes
    -----
    - This function depends on `get_team_per_version`, `get_occur_version`,
      and `get_graph`. It is the highest-level pipeline function.
    - Returned graphs can be saved with pickle or visualized with NetworkX.
    """

    possible_teams = get_team_per_version(version, char_names)
    co_occur = get_occur_version(possible_teams, image_english_mapping)
    return get_graph(co_occur)

In [43]:
os.makedirs("graphs", exist_ok=True)

for i in range(52, 1, -1):
    filename = f"data/graphs/graph_{i}.pickle"

    # If the file already exists, skip it
    if os.path.exists(filename):
        print(f"Already exists, skipped: {filename}")
        continue

    # Otherwise, create and save the graph
    graph = create_graph(i)
    with open(filename, "wb") as f:
        pickle.dump(graph, f)

    print(f"Created and saved: {filename}")

Already exists, skipped: data/graphs/graph_52.pickle
Already exists, skipped: data/graphs/graph_51.pickle
Already exists, skipped: data/graphs/graph_50.pickle
Already exists, skipped: data/graphs/graph_49.pickle
Already exists, skipped: data/graphs/graph_48.pickle
Already exists, skipped: data/graphs/graph_47.pickle
Already exists, skipped: data/graphs/graph_46.pickle
Already exists, skipped: data/graphs/graph_45.pickle
Already exists, skipped: data/graphs/graph_44.pickle
Already exists, skipped: data/graphs/graph_43.pickle
Already exists, skipped: data/graphs/graph_42.pickle
Already exists, skipped: data/graphs/graph_41.pickle
Already exists, skipped: data/graphs/graph_40.pickle
Already exists, skipped: data/graphs/graph_39.pickle
Already exists, skipped: data/graphs/graph_38.pickle
Already exists, skipped: data/graphs/graph_37.pickle
Already exists, skipped: data/graphs/graph_36.pickle
Already exists, skipped: data/graphs/graph_35.pickle
Already exists, skipped: data/graphs/graph_34.

# Getting the data per node

In [44]:
API_URL = "https://gi.yatta.moe/api/v2/en/avatar"


def fetch_and_print_json(url):
    """
    Fetches JSON data from a given URL, converts the character data
    into a pandas DataFrame, and prints the DataFrame.
    """
    print(f"--- Fetching data from: {url} ---")

    # Make the GET request
    response = requests.get(url)

    # Raise an HTTPError for bad responses (4xx or 5xx)
    response.raise_for_status()

    # Get the JSON data
    data = response.json()

    # Check if the 'items' key exists in the nested 'data' object
    if "data" in data and "items" in data["data"]:
        character_items = data["data"]["items"]

        # Convert the dictionary of characters (where keys are IDs)
        # into a pandas DataFrame.
        # orient='index' tells pandas that the keys of the dictionary (character IDs)
        # should become the DataFrame index.
        df = pd.DataFrame.from_dict(character_items, orient="index")

        return df

    else:
        return print("\n--- Raw JSON Data (Could not extract character items) ---")

    # Ensure you have the 'requests' and 'pandas' libraries installed:
    # pip install requests pandas


df_characteristics = fetch_and_print_json(API_URL)

--- Fetching data from: https://gi.yatta.moe/api/v2/en/avatar ---


In [52]:
# Getting the Player Engagement thru Abyss Rank
data_list = []

file_path = 'data/abyss_rank_activity.csv'

if not os.path.exists(file_path):
    data_list = []

    for version in range(52, 0, -1):  # from 52 down to 1
        url = f'https://api.yshelper.com/ys/getAbyssRank.php?star=all&role=all&lang=en&version={version}'
        response = requests.get(url).json()
        version_name = response['version']
        tips = response['tips']

        # Extract Total and Effective samples
        total_match = re.search(r'Total\s*(\d+)', tips)
        effective_match = re.search(r'effective\s*(\d+)', tips, re.IGNORECASE)

        total_samples = int(total_match.group(1)) if total_match else None
        effective_samples = int(effective_match.group(1)) if effective_match else None

        print(f"Version {version}: Total={total_samples}, Effective={effective_samples}")

        data_list.append({
            'Version': version,
            'Version Name': version_name,
            'Total Samples': total_samples,
            'Effective Samples': effective_samples
        })
    
    df = pd.DataFrame(data_list)
    os.makedirs('data', exist_ok=True)
    df.to_csv(file_path, index=False)
else:
    print(f"{file_path} already exists. Skipping data scraping.")
    df = pd.read_csv(file_path)

# Clean the data further
samples = ['Total Samples', 'Effective Samples']
for sample_columns in samples:
    df[sample_columns] = pd.to_numeric(df[sample_columns])

df['Patch'] = df['Version Name'].str[9:12]

# I used regex so that i can eventually path label
df['Patch Label'] = df['Version Name'].str.extract(r'(\d\.\d)\(Phase\s+(\w*)\)')\
                                                         .apply(lambda x: ' '.join(x), axis=1)

df = df.iloc[::-1]

df.to_csv('data/abyss_rank_activity.csv', index=False)

data/abyss_rank_activity.csv already exists. Skipping data scraping.


# Getting the Predicted Data

In [47]:
# Load webpage
url = "https://genshin-impact.fandom.com/wiki/Character_Role"
response = requests.get(url)
response.raise_for_status()

# Parse HTML
soup = BeautifulSoup(response.text, "html.parser")

# Select ALL tables with the fandom CSS class
tables = soup.find_all("table", class_="article-table")

if len(tables) < 2:
    raise ValueError("Could not find more than one matching table.")

# Use the *second table* on the page
target_table = tables[1]

df_target = pd.read_html(StringIO(str(target_table)))[0]

# Save as CSV
output_path = "data/genshin_character_roles.csv"

targets = df_target.columns[2:]
targets

for target in targets:
    df_target[target] = df_target[target].apply(lambda x: {"✘": 0, "✔": 1}[x])
df_target.to_csv(output_path, index=False)

# Combine them all

In [48]:
# Characteristics df
problematic_char = list(set(df_characteristics["name"]) - set(df_target["Name"]))
is_problematic_char = df_characteristics["name"].isin(problematic_char)
df_characteristics_correct = df_characteristics[~is_problematic_char]

# Target df
problematic_target = list(set(df_target["Name"]) - set(df_characteristics["name"]))
is_problematic_target = df_target["Name"].isin(problematic_target)
df_target_correct = df_target[~is_problematic_target]

In [49]:
df_all = df_characteristics_correct.merge(
    df_target_correct, left_on="name", right_on="Name"
)

df_all_cleaned = df_all.rename(columns={"Surviv­ability": "Survivability"})
relevant_cols = [
    "name",
    "element",
    "weaponType",
    "region",
    "specialProp",
    "bodyType",
    "Element",
    "On-Field",
    "Off-Field",
    "DPS",
    "Support",
    "Survivability",
]
df_all_cleaned[relevant_cols].to_csv("data/cleaned_node.csv", index=False)

In [50]:
df_all_cleaned = pd.read_csv(
    "data/cleaned_node.csv",
)

# Clean the graphs

In [51]:
df_all_cleaned = pd.read_csv(
    "data/cleaned_node.csv",
)

with open("data/graphs/graph_33.pickle", "rb") as f:
    G = pickle.load(f)

graph_names = set(G.nodes)